### Todo's
- On cell 6 where the multiple folds are trained, we can create an averaging of them to create the ensemble algorithms which should improve performance

In [1]:
import pm4py

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.model_selection import GroupShuffleSplit



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




In [2]:
DomesticDeclarations_URL = '/Users/laimis/Documents/Dissertation/BPI_Challenge_2020/Training for Dissertation/Data/DomesticDeclarations.xes.gz'

def DomesticDeclarations_cleaning(url):
    df = pm4py.read_xes(url)
    df = pm4py.format_dataframe(
        df,
        case_id='case:concept:name',
        activity_key='concept:name',
        timestamp_key='time:timestamp'
    )

    def transform_unknowns():
        s1 = df['case:DeclarationNumber']
        s2 = pd.Series(index=s1.index, dtype=s1.dtype)

        instance = 0
        for i in range(len(s1)):
            if s1[i] == 'UNKNOWN':
                s2[i] = str(f'UNKNOWN {instance}')
                if i + 1 < len(s1) and s1[i + 1] != 'UNKNOWN':
                    instance += 1
            else:
                s2.loc[i] = s1.loc[i]
        return s2

    df['case:DeclarationNumber'] = transform_unknowns()

    df = df[['case:Amount', 'org:role', 'time:timestamp', 'org:resource', 'concept:name', 'case:DeclarationNumber', 'case:concept:name']]

    trace_start = df.groupby('case:DeclarationNumber')['time:timestamp'].transform('min')

    # Time intervals in seconds
    df['time_prev'] = (df.groupby('case:DeclarationNumber')['time:timestamp']
                       .diff()
                       .dt.total_seconds()
                       .fillna(0))
    
    df['time_start'] = (df['time:timestamp'] - trace_start).dt.total_seconds()
    
    # Target converted directly to days
    trace_end = df.groupby('case:DeclarationNumber')['time:timestamp'].transform('max')
    df['remaining_time (y)'] = (trace_end - df['time:timestamp']).dt.total_seconds() / 86400.0

    df['step_counter'] = df.groupby('case:DeclarationNumber').cumcount()
    df['day of week'] = df['time:timestamp'].apply(lambda s_: s_.weekday())

    df = df[['case:Amount', 'org:role', 'day of week',
             'org:resource', 'concept:name', 'case:DeclarationNumber',
             'time_prev', 'time_start', 'step_counter', 'case:concept:name',
             'remaining_time (y)']]

    return df.reset_index(drop=True)

df = DomesticDeclarations_cleaning(DomesticDeclarations_URL)
df

/opt/anaconda3/lib/python3.13/site-packages/pm4py/utils.py:1005: UserWarning: In the current version, the import/export operation uses `r4pm` by default for importing/exporting files faster.
  warnings.warn(


,case:Amount,org:role,day of week,org:resource,concept:name,case:DeclarationNumber,time_prev,time_start,step_counter,case:concept:name,remaining_time (y)
0,600.844121,EMPLOYEE,1,STAFF MEMBER,Declaration SUBMITTED by EMPLOYEE,declaration number 100001,0.0,0.0,0,declaration 100000,13.341123
1,600.844121,ADMINISTRATION,2,STAFF MEMBER,Declaration APPROVED by ADMINISTRATION,declaration number 100001,693519.0,693519.0,1,declaration 100000,5.314282
2,600.844121,SUPERVISOR,3,STAFF MEMBER,Declaration FINAL_APPROVED by SUPERVISOR,declaration number 100001,90019.0,783538.0,2,declaration 100000,4.272396
3,600.844121,UNDEFINED,4,SYSTEM,Request Payment,declaration number 100001,92624.0,876162.0,3,declaration 100000,3.200359
4,600.844121,UNDEFINED,0,SYSTEM,Payment Handled,declaration number 100001,276511.0,1152673.0,4,declaration 100000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
56432,26.218933,EMPLOYEE,0,STAFF MEMBER,Declaration SUBMITTED by EMPLOYEE,declaration number 99996,0.0,0.0,0,declaration 99995,2.855775
56433,26.218933,ADMINISTRATION,0,STAFF MEMBER,Declaration APPROVED by ADMINISTRATION,declaration number 99996,11.0,11.0,1,declaration 99995,2.855648
56434,26.218933,SUPERVISOR,1,STAFF MEMBER,Declaration FINAL_APPROVED by SUPERVISOR,declaration number 99996,61169.0,61180.0,2,declaration 99995,2.147674
56435,26.218933,UNDEFINED,2,SYSTEM,Request Payment,declaration number 99996,78867.0,140047.0,3,declaration 99995,1.234861


In [3]:
X = df.copy()
y = df['remaining_time (y)']

num_cols = ['case:Amount', 'time_prev', 'time_start', 'step_counter', 'day of week']
cat_cols = ['org:role', 'org:resource', 'concept:name']

X_onehot = (df[['org:role', 'org:resource', 'concept:name']])
X_onehot = (pd.get_dummies(X_onehot, columns=['org:role', 'org:resource', 'concept:name'], drop_first=True))
X_unified = pd.concat([df[num_cols], X_onehot], axis=1, join='inner')

In [4]:
# %%
# 1. Feature Engineering: Include trace prefix activity counts
activity_dummies = pd.get_dummies(df['concept:name'], prefix='act')
prefix_counts = activity_dummies.groupby(df['case:DeclarationNumber']).cumsum()

# Combine numeric features, one-hot current event features, and prefix history
X_unified = pd.concat([df[num_cols], X_onehot, prefix_counts], axis=1)
y = df['remaining_time (y)']

# 2. Group Split by Case
groups = df['case:DeclarationNumber']
gss = GroupShuffleSplit(n_splits=4, train_size=0.7, random_state=42)
train_idx, test_idx = next(gss.split(X_unified, y, groups=groups))

In [ ]:
# for fold, (train_idx, test_idx) in enumerate(gss.split(X_unified, y, groups=groups)):
#     print(f"Fold {fold}: Train size = {len(train_idx)}, Test size = {len(test_idx)}")

Fold 0: Train size = 39480, Test size = 16957
Fold 1: Train size = 39482, Test size = 16955
Fold 2: Train size = 39567, Test size = 16870
Fold 3: Train size = 39508, Test size = 16929


In [6]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error

results = []

models = {
    "Ridge Regression": (Ridge(alpha=100.0, random_state=42), True),
    "Random Forest":    (RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1), False),
    "HistGradientBoosting": (HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08, random_state=42), False),
}

for name, (model, use_scaled) in models.items():
    print(f'Training {name}')

    for fold, (train_idx, test_idx) in enumerate(gss.split(X_unified, y, groups=groups)):
        
        # 1. Slice data (adjust names to match your scaled/unscaled variables)
        X_tr = X_unified.iloc[train_idx]
        X_te = X_unified.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # 2. Fit and predict
        model.fit(X_tr, y_train)
        y_pred = np.clip(model.predict(X_te), a_min=0, a_max=None)

        # 3. Evaluate
        results.append({
            "Model": name,
            "Fold": fold,
            "MAE": mean_absolute_error(y_test, y_pred),
            "RMSE": root_mean_squared_error(y_test, y_pred),
            "R2": r2_score(y_test, y_pred)
        })

print(pd.DataFrame(results).to_string(index=False))

Training Ridge Regression
Training Random Forest
Training HistGradientBoosting
               Model  Fold      MAE      RMSE       R2
    Ridge Regression     0 3.957942  9.930719 0.169213
    Ridge Regression     1 3.791919  9.918262 0.159587
    Ridge Regression     2 3.871092 11.078580 0.133633
    Ridge Regression     3 3.856421 10.939543 0.134540
       Random Forest     0 3.894646 10.258273 0.113503
       Random Forest     1 3.737449 10.349694 0.084883
       Random Forest     2 3.831330 11.470009 0.071330
       Random Forest     3 3.777349 11.160966 0.099151
HistGradientBoosting     0 3.874788  9.926661 0.169891
HistGradientBoosting     1 3.702180  9.902054 0.162331
HistGradientBoosting     2 3.809092 11.124916 0.126371
HistGradientBoosting     3 3.764643 10.948489 0.133124
